# 02 — CTE Correction for FLT-only ACS Exposures

Some ACS/WFC exposures were taken in a window where the CalACS pipeline did not
produce FLC files (CTE-corrected). These need to be run through CalACS manually
to produce FLC files before alignment and drizzling.

Affected cases (FLT files with no FLC partner):
- W2M0043+0052 — F814W
- W2M1106+0221 — F814W
- W2M2152-0051 — F475W and F814W

This notebook:
1. Identifies FLT-only exposures in `data_acs/raw/`
2. Downloads their corresponding RAW files from MAST
3. Runs CalACS to produce FLC files in a staging directory
4. Moves FLC outputs into `data_acs/raw/<target>/` alongside the existing files

In [ ]:
import os
import shutil
import subprocess
import yaml
from pathlib import Path

# CRDS env vars must be set before importing crds — the module reads them at import time
CRDS_CACHE = Path.home() / 'crds_cache'
CRDS_CACHE.mkdir(exist_ok=True)
os.environ['CRDS_SERVER_URL']  = 'https://hst-crds.stsci.edu'
os.environ['CRDS_PATH']        = str(CRDS_CACHE)
os.environ['CRDS_OBSERVATORY'] = 'hst'

import crds
from astropy.io import fits
from astroquery.mast import Observations
from acstools import acs_destripe_plus
from stwcs import updatewcs

with open('../config/wfc3_ir_drizzle_params.yaml') as f:
    cfg = yaml.safe_load(f)

dl_cfg = cfg['acs_download']
RAW_DIR     = Path('../data_acs/raw')
STAGING_DIR = Path('../data_acs/cte_staging')
STAGING_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Bootstrap the CRDS cache if it hasn't been initialized yet.
# CRDS needs a server_config file before getreferences() will work.
# crds sync without --fetch-references downloads only context metadata (~few MB),
# not any reference files — those are pulled on demand in cell 5.

server_config = CRDS_CACHE / 'config' / 'hst' / 'server_config'
if not server_config.exists():
    print('Initializing CRDS cache (one-time, downloads context metadata only) ...')
    subprocess.run(
        ['/opt/miniforge3/envs/stenv/bin/crds', 'sync',
         '--contexts', 'hst-operational'],
        env=os.environ.copy(),
        check=True
    )
    print('CRDS cache initialized.')
else:
    print('CRDS cache already initialized.')

# jref points to where CRDS caches ACS reference files so that header entries
# like "jref$78f1831ej_drk.fits" resolve to the correct local path
jref_dir = CRDS_CACHE / 'references' / 'hst' / 'acs'
jref_dir.mkdir(parents=True, exist_ok=True)
os.environ['jref'] = str(jref_dir) + '/'

print(f'CRDS cache : {CRDS_CACHE}')
print(f'jref       : {os.environ["jref"]}')

In [ ]:
# Find every FLT file that has no paired FLC (same rootname, _flc.fits suffix)
# These are the exposures that need CTE correction

flt_only = []
for f in sorted(RAW_DIR.glob('*/*_flt.fits')):
    if f.name.startswith('hst_'):
        continue
    flc_partner = f.parent / f.name.replace('_flt.fits', '_flc.fits')
    if not flc_partner.exists():
        flt_only.append(f)

# Rootname is the filename stem without the _flt suffix (e.g. 'jd5k11jlq')
flt_only_rootnames = {f.name.replace('_flt.fits', '') for f in flt_only}

print(f'FLT-only files needing CTE correction: {len(flt_only)}')
print(f'Unique rootnames: {len(flt_only_rootnames)}')
print()
from collections import Counter
from astropy.io import fits as afits

def get_filter(hdr):
    f1, f2 = hdr.get('FILTER1', ''), hdr.get('FILTER2', '')
    return f1 if 'CLEAR' not in f1 else f2

summary = Counter()
for f in flt_only:
    with afits.open(f) as h:
        filt = get_filter(h[0].header)
    summary[(f.parent.name, filt)] += 1
for (target, filt), n in sorted(summary.items()):
    print(f'  {target}  {filt}  {n} files')

In [ ]:
# Query MAST for all ACS/WFC products from proposal 14706,
# then filter down to RAW files matching only the FLT-only rootnames

obs_table = Observations.query_criteria(
    proposal_id=dl_cfg['proposal_id'],
    instrument_name=dl_cfg['instrument_name'],
    project=dl_cfg['project'],
)
all_products = Observations.get_product_list(obs_table)

# Keep only RAW science files whose filename contains one of our target rootnames
raw_products = Observations.filter_products(
    all_products,
    productSubGroupDescription='RAW',
    productType='SCIENCE',
)
raw_products = raw_products[
    [any(rn in row['productFilename'] for rn in flt_only_rootnames)
     for row in raw_products]
]

print(f'RAW files to download: {len(raw_products)}')

# Download into the staging directory; cache=True skips files already present
Observations.download_products(raw_products, download_dir=str(STAGING_DIR), cache=True)

In [ ]:
# Move RAW files from the MAST staging tree into the flat STAGING_DIR
# so CalACS can find its reference files relative to a single working directory
# download_dir places files under <download_dir>/mastDownload/HST/<rootname>/

for raw in sorted(STAGING_DIR.glob('mastDownload/HST/*/*_raw.fits')):
    dest = STAGING_DIR / raw.name
    if not dest.exists():
        shutil.move(str(raw), str(dest))

# Clean up the mastDownload subdirectory tree left by astroquery
mast_sub = STAGING_DIR / 'mastDownload'
if mast_sub.exists():
    shutil.rmtree(str(mast_sub))

raw_files = sorted(STAGING_DIR.glob('*_raw.fits'))
print(f'RAW files ready for CalACS: {len(raw_files)}')

In [ ]:
import requests

CRDS_DOWNLOAD_BASE = 'https://hst-crds.stsci.edu/unchecked_get/references/hst'

# All calibration reftypes that appear in ACS RAW headers with jref$ values.
# Earlier failed runs may have written corrupted values (e.g. 'jref$a') to some
# of these keywords; fetching all of them from CRDS repairs any corrupted entries.
ACS_CAL_REFTYPES = [
    'biasfile', 'darkfile', 'pfltfile', 'ccdtab', 'bpixtab',
    'oscntab', 'crrejtab', 'snkcfile', 'satufile',
    'pctetab', 'drkcfile',
]

def ensure_jref_files(header, jref_dir):
    """Download any jref$ FITS file in header not already in jref_dir.
    Silently skips values that are not valid FITS filenames (corrupted entries)."""
    jref_dir = Path(jref_dir)
    for val in header.values():
        if not isinstance(val, str) or not val.startswith('jref$'):
            continue
        fname = val.split('$', 1)[1]
        if not fname.endswith('.fits') or len(fname) < 12:
            continue  # skip corrupted entries like 'jref$a'
        dest = jref_dir / fname
        if dest.exists():
            continue
        url = f'{CRDS_DOWNLOAD_BASE}/{fname}'
        print(f'    Downloading {fname} ...')
        r = requests.get(url, stream=True, timeout=120)
        r.raise_for_status()
        with open(dest, 'wb') as fh:
            for chunk in r.iter_content(chunk_size=65536):
                fh.write(chunk)

for raw in sorted(STAGING_DIR.glob('*_raw.fits')):
    flc_out = STAGING_DIR / raw.name.replace('_raw.fits', '_flc.fits')
    if flc_out.exists():
        print(f'{raw.name}: FLC already exists, skipping')
        continue

    print(f'Processing {raw.name} ...')

    with fits.open(str(raw)) as hdul:
        params = dict(hdul[0].header)
    # Tell CRDS that CTE correction is requested so it includes pctetab + drkcfile
    params['PCTECORR'] = 'PERFORM'

    # Fetch ALL calibration reference files; this overwrites any corrupted values
    # written by previous failed runs (e.g. BIASFILE = 'jref$a')
    refs = crds.getreferences(params, reftypes=ACS_CAL_REFTYPES, observatory='hst')

    with fits.open(str(raw), mode='update') as hdul:
        for reftype, filepath in refs.items():
            hdul[0].header[reftype.upper()] = 'jref$' + Path(filepath).name
        hdul[0].header['PCTECORR'] = 'PERFORM'

    # Download any jref$ files not yet in the local cache (covers WCS/geometric refs
    # like IDCTAB, DGEOFILE, etc. that aren't in ACS_CAL_REFTYPES above)
    with fits.open(str(raw)) as hdul:
        ensure_jref_files(hdul[0].header, jref_dir)

    acs_destripe_plus.destripe_plus(str(raw), cte_correct=True)
    status = 'OK' if flc_out.exists() else 'FAILED — no FLC produced'
    print(f'  {raw.name} → {status}')

In [ ]:
# Move FLC outputs from staging into data_acs/raw/<TARGNAME>/
# alongside the existing FLT files
# Then run updatewcs on each moved file — subarray FLC outputs from
# acs_destripe_plus don't carry proper WCS until updatewcs is called
# Finally clean up remaining staging files

newly_moved = []
for flc in sorted(STAGING_DIR.glob('*_flc.fits')):
    target = fits.getval(str(flc), 'TARGNAME', ext=0)
    dest = RAW_DIR / target / flc.name
    if dest.exists():
        print(f'{flc.name}: already in raw dir, skipping')
    else:
        shutil.move(str(flc), str(dest))
        newly_moved.append(dest)

# Remove remaining staging files (RAW + acs_destripe_plus FLT outputs)
for f in STAGING_DIR.glob('*.fits'):
    f.unlink()

print(f'Moved {len(newly_moved)} FLC files to data_acs/raw/')
print('Staging directory cleared.')
print()

# Update WCS on all newly created FLC files
if newly_moved:
    print(f'Updating WCS on {len(newly_moved)} files ...')
    for flc_path in newly_moved:
        print(f'  {flc_path.name}')
        updatewcs.updatewcs(str(flc_path))
    print('WCS update complete.')
else:
    print('No new FLC files to update.')

In [ ]:
# Remove files that calibration tools dump into the notebooks/ directory:
#   *_hlet.fits  — headerlet files written by updatewcs
#   *.log        — CalACS / TweakReg / headerlet logs
#   shifts_wcs.fits — TweakReg residual file
#   mastDownload/   — leftover MAST download tree from earlier runs

import shutil

NB_DIR = Path('.')

removed = []

for pattern in ('*_hlet.fits', '*.log', 'shifts_wcs.fits'):
    for f in NB_DIR.glob(pattern):
        f.unlink()
        removed.append(f.name)

mast_sub = NB_DIR / 'mastDownload'
if mast_sub.exists():
    shutil.rmtree(str(mast_sub))
    removed.append('mastDownload/')

print(f'Removed {len(removed)} items from notebooks/:')
for name in sorted(removed):
    print(f'  {name}')

In [ ]:
# Verify that every target/filter pair now has FLC files in data_acs/raw/.
# After CTE correction, all combinations — including those that started as FLT-only —
# should have at least one FLC file ready for alignment and drizzling.

from astropy.table import Table
from collections import defaultdict

def get_filter(hdr):
    f1, f2 = hdr.get('FILTER1', ''), hdr.get('FILTER2', '')
    return f1 if 'CLEAR' not in f1 else f2

flc_counts = defaultdict(int)
flt_counts = defaultdict(int)

for f in sorted(RAW_DIR.glob('*/*fl?.fits')):
    if f.name.startswith('hst_'):
        continue
    with fits.open(f) as hdul:
        filt = get_filter(hdul[0].header)
    key = (f.parent.name, filt)
    if f.name.endswith('_flc.fits'):
        flc_counts[key] += 1
    else:
        flt_counts[key] += 1

expected = cfg['acs_targets']
rows = []
all_ok = True

for target in sorted(expected):
    for filt in sorted(expected[target]):
        key = (target, filt)
        orig_type = expected[target][filt]
        n_flc = flc_counts[key]
        n_flt = flt_counts[key]
        ok = n_flc > 0
        status = 'OK' if ok else 'MISSING FLC'
        if not ok:
            all_ok = False
        rows.append((target, filt, orig_type, n_flc, n_flt, status))

t = Table(rows=rows,
          names=['Target', 'Filter', 'Original', 'N_FLC', 'N_FLT', 'Status'])
t.pprint(max_lines=-1, max_width=100)
print()
if all_ok:
    print('All targets have FLC files — ready for alignment.')
else:
    print('WARNING: some targets are still missing FLC files.')